# 04f · Feature representation guide (Section 9)

**Purpose.** Fill the decision table (situation → recommended representation → evidence) from the results, and write the data setup checklist: baseline correction, split by session, train only fitting, baseline only sanity check.

**Inputs**
- `results/*.csv` and `report/tables/` from 04a to 04e

**Outputs**
- `report/tables/` guide table

## Colab setup
Run this first on Google Colab (Runtime → Change runtime type → **GPU**). Before running, upload
**`results.zip`** (results of notebooks 01-03) to `/content` with the Files panel on the left. The cell clones the
repo, downloads the processed data from the shared Drive folder, unpacks the results, and moves into this
notebook's folder so the rest runs unchanged. On a laptop it does nothing.

In [1]:
import glob
import os
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO = "/content/gas-repr-guide"
REPO_URL = "https://github.com/YasserAlbogami/gas-repr-guide.git"
DATA_URL = "https://drive.google.com/drive/folders/1rNIcZkHDFhH2dMVq8TgNWt_0IH2yw2_I"   # processed data (Yasser)
DATA_DIRS = ["gas_wind_tunnel", "everyday_scents"]

if IN_COLAB:
    # 1. code (src/). Tried in this order:
    #    a) clone the GitHub repo (works when the repo is public)
    #    b) clone with a GitHub token saved in Colab Secrets (key icon on the left) as GITHUB_TOKEN
    #    c) src.zip uploaded to /content (Files panel on the left)
    if not os.path.exists(f"{REPO}/src"):
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")           # fail instead of asking for a password
        ok = subprocess.run(["git", "clone", "-q", REPO_URL, REPO], env=env).returncode == 0
        if not ok:
            token = None
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                pass
            if token:
                url = REPO_URL.replace("https://", f"https://{token}@")
                ok = subprocess.run(["git", "clone", "-q", url, REPO], env=env).returncode == 0
        if not ok and os.path.exists("/content/src.zip"):
            os.makedirs(f"{REPO}/notebooks/03_encoder", exist_ok=True)
            shutil.unpack_archive("/content/src.zip", REPO)
            ok = True
        if not ok:
            raise RuntimeError("Could not get the code: the repo is private. Make it public, or add GITHUB_TOKEN "
                               "in Colab Secrets, or upload src.zip to /content, then run this cell again.")
    elif os.path.exists(f"{REPO}/.git"):
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
    os.makedirs(f"{REPO}/results", exist_ok=True)

    # 2. processed data (only downloaded once per Colab session)
    processed = f"{REPO}/data/processed"
    if not all(os.path.exists(f"{processed}/{d}/X_signals.npy") for d in DATA_DIRS):
        import gdown
        tmp = "/content/processed_download"
        gdown.download_folder(DATA_URL, output=tmp, quiet=False)
        for d in DATA_DIRS:
            found = glob.glob(f"{tmp}/**/{d}", recursive=True)
            assert found, f"folder {d} not found in the Drive download"
            shutil.copytree(found[0], f"{processed}/{d}", dirs_exist_ok=True)
    for d in DATA_DIRS:
        files = sorted(os.listdir(f"{processed}/{d}"))
        print(d, files)
        for f in ["X_signals.npy", "meta.csv", "info.json"]:
            assert f in files, f"{d}/{f} is missing"

    # 2b. results of the earlier notebooks (01, 02, 03 and the 04 ones already run), from results.zip
    #     uploaded to /content (Files panel on the left). The zip holds a results/ folder (+ report/tables/).
    if os.path.exists("/content/results.zip"):
        shutil.unpack_archive("/content/results.zip", REPO)
    have = sorted(os.path.basename(f) for f in glob.glob(f"{REPO}/results/*.csv"))
    print("results:", have)
    main = ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted",
            "03a_gwt_encoder", "03b_es_encoder", "03c_end_to_end_reference"]
    missing = [n for n in main if f"{n}.csv" not in have]
    assert not missing, f"upload results.zip to /content first (missing: {missing})"

    # 3. run the rest of the notebook from its own folder, as on a laptop (so ../.. = project root)
    os.chdir(f"{REPO}/notebooks/04_comparisons")

    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
          else "NONE, switch on Runtime > Change runtime type > GPU")

In [2]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from sklearn.ensemble import RandomForestClassifier
from src.cv import load_baseline_level, load_processed, make_folds
from src.features.handcrafted import FEATURE_NAMES, handcrafted_features

TABLES = ROOT / "report/tables"
FIG = ROOT / "report/figures"
RESULTS = ROOT / "results"
MAIN = ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted",
        "03a_gwt_encoder", "03b_es_encoder", "03c_end_to_end_reference"]
missing = [n for n in MAIN if not (RESULTS / f"{n}.csv").exists()]
assert not missing, f"run these notebooks first: {missing}"
scores = pd.concat([pd.read_csv(RESULTS / f"{n}.csv") for n in MAIN], ignore_index=True)
REPS = ["R1_raw", "R2_handcrafted", "R3_encoder"]
HEADS = ["logreg", "svm", "rf", "mlp"]
cost = pd.read_csv(TABLES / "C4_cost.csv")
sanity = pd.read_csv(RESULTS / "04d_baseline_sanity_check.csv")
eff = pd.read_csv(RESULTS / "04b_label_efficiency.csv")

## Best representation per dataset and task
Score = best head (highest mean macro F1 over 3 seeds × 5 folds).

In [3]:
mean_f1 = scores[scores["rep"].isin(REPS)].groupby(["dataset", "task", "rep", "head"])["macro_f1"].mean()
best_rep = mean_f1.groupby(["dataset", "task", "rep"]).max().unstack("rep")[REPS]
best_rep["winner"] = best_rep.idxmax(axis=1)
best_rep.round(3)

rep             R1_raw  R2_handcrafted  R3_encoder          winner
dataset task                                                      
es      binary   0.953           0.962       0.964      R3_encoder
        multi    0.817           0.871       0.856  R2_handcrafted
gwt     binary   1.000           1.000       0.997          R1_raw
        multi    0.992           0.996       0.987  R2_handcrafted

## Interpretability: random forest importance of the R2 features
Fitted on the **training part** of fold 0, seed 0 only (no test data).

In [4]:
importance = {}
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)
    F = handcrafted_features(X, load_baseline_level(dataset), info["n_base"], info["exposure_end"])  # (N, C * 12)
    tr, _ = make_folds(dataset, meta, seed=0)[0]
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0).fit(F[tr], meta["y_multi"].iloc[tr])
    names = [f"{info.get('channels', [f's{c}' for c in range(X.shape[1])])[c]}:{f}" for c in range(X.shape[1]) for f in FEATURE_NAMES]
    imp = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False)
    importance[dataset] = imp
    by_feature = imp.groupby(lambda n: n.split(":")[1]).sum().sort_values(ascending=False)
    print(dataset, "- top 10 sensor:feature"); display(imp.head(10).round(4))
    print(dataset, "- importance summed per feature type"); display(by_feature.round(3))

gwt - top 10 sensor:feature


s35:ema_max_0.01      0.0163
s32:ema_min_0.01      0.0161
s32:baseline_level    0.0159
s35:auc               0.0156
s35:ema_max_0.1       0.0153
s2:baseline_level     0.0153
s0:baseline_level     0.0152
s32:ema_min_0.1       0.0143
s32:ema_max_0.1       0.0140
s28:baseline_level    0.0129
dtype: float64

gwt - importance summed per feature type


baseline_level    0.415
auc               0.103
ema_max_0.1       0.086
ema_max_0.01      0.085
ema_min_0.1       0.080
peak              0.064
steady_state      0.053
normalized        0.047
ema_min_0.01      0.036
rise_slope        0.022
time_to_peak      0.006
decay_slope       0.003
dtype: float64

es - top 10 sensor:feature


co:normalized         0.0385
no2:ema_max_0.01      0.0370
smoke:ema_max_0.01    0.0356
co:steady_state       0.0334
smoke:auc             0.0322
no2:steady_state      0.0321
smoke:peak            0.0301
smoke:normalized      0.0277
co:ema_min_0.01       0.0274
smoke:steady_state    0.0268
dtype: float64

es - importance summed per feature type


ema_max_0.01      0.138
steady_state      0.129
normalized        0.120
auc               0.120
ema_max_0.1       0.115
peak              0.108
rise_slope        0.066
baseline_level    0.061
ema_min_0.01      0.051
ema_min_0.1       0.038
decay_slope       0.029
time_to_peak      0.025
dtype: float64

## The guide (Section 9), filled from the results

In [5]:
def winner(dataset, task):
    return best_rep.loc[(dataset, task), "winner"]


def f1(dataset, task, rep):
    return best_rep.loc[(dataset, task), rep]


# cheapest representation within 0.02 macro F1 of the best, per dataset (multiclass)
cheap = {}
for dataset in ["gwt", "es"]:
    ok = [r for r in REPS if f1(dataset, "multi", r) >= f1(dataset, "multi", winner(dataset, "multi")) - 0.02]
    c = cost[(cost["dataset"] == dataset) & cost["rep"].isin(ok)]
    cheap[dataset] = c.assign(total=c["build_features_s"] + c["train_best_head_s"]).sort_values("total")["rep"].iloc[0]

low = eff[eff["train_fraction"] == eff["train_fraction"].min()]
low_winner = low.groupby(["rep", "head"])["macro_f1"].mean().groupby("rep").max().idxmax()

guide = pd.DataFrame([
    ("Lab data, many labeled recordings", winner("gwt", "multi"), "C1 GasWindTunnel, C3",
     f"GWT multiclass best: {f1('gwt', 'multi', winner('gwt', 'multi')):.3f}; with 10% labels the best is {low_winner}"),
    ("Real world scents, few samples", winner("es", "multi"), "C1 EverydayScents, C2",
     f"ES multiclass best: {f1('es', 'multi', winner('es', 'multi')):.3f}"),
    ("Need interpretable features", "R2_handcrafted", "C1, random forest importance",
     f"R2 vs best: GWT {f1('gwt', 'multi', 'R2_handcrafted'):.3f} vs {f1('gwt', 'multi', winner('gwt', 'multi')):.3f}, "
     f"ES {f1('es', 'multi', 'R2_handcrafted'):.3f} vs {f1('es', 'multi', winner('es', 'multi')):.3f}"),
    ("No GPU, limited time", f"GWT: {cheap['gwt']}, ES: {cheap['es']}", "C4",
     "cheapest representation within 0.02 macro F1 of the best"),
    ("Detecting one target (binary)", f"GWT: {winner('gwt', 'binary')}, ES: {winner('es', 'binary')}", "C1 binary tables", ""),
    ("Identifying many classes", winner("gwt", "multi"), "C1 multiclass tables", "10-class GasWindTunnel task"),
], columns=["Your situation", "Recommended representation", "Evidence", "Numbers"])
guide.to_csv(TABLES / "guide_table.csv", index=False)
guide

,Your situation,Recommended representation,Evidence,Numbers
0,"Lab data, many labeled recordings",R2_handcrafted,"C1 GasWindTunnel, C3",GWT multiclass best: 0.996; with 10% labels th...
1,"Real world scents, few samples",R2_handcrafted,"C1 EverydayScents, C2",ES multiclass best: 0.871
2,Need interpretable features,R2_handcrafted,"C1, random forest importance","R2 vs best: GWT 0.996 vs 0.996, ES 0.871 vs 0.871"
3,"No GPU, limited time","GWT: R1_raw, ES: R2_handcrafted",C4,cheapest representation within 0.02 macro F1 o...
4,Detecting one target (binary),"GWT: R1_raw, ES: R3_encoder",C1 binary tables,
5,Identifying many classes,R2_handcrafted,C1 multiclass tables,10-class GasWindTunnel task


## Data setup checklist

In [6]:
chance = {("gwt", "multi"): 0.10, ("es", "multi"): 0.25, ("gwt", "binary"): 0.5, ("es", "binary"): 0.5}
leak = sanity.groupby(["dataset", "task", "rep", "head"])["balanced_acc"].mean().groupby(["dataset", "task"]).max()
checklist = [
    "Baseline correction: subtract the per-sensor resting level before building features.",
    "Split by session (grouped folds) whenever several samples come from one session.",
    "Fit every scaler, encoder and head on the training fold only; tune on an inner split.",
    "Run the baseline only sanity check: " + "; ".join(
        f"{d} {t} {v:.2f} (chance {chance[(d, t)]:.2f})" for (d, t), v in leak.items()),
]
for item in checklist:
    print("-", item)

- Baseline correction: subtract the per-sensor resting level before building features.
- Split by session (grouped folds) whenever several samples come from one session.
- Fit every scaler, encoder and head on the training fold only; tune on an inner split.
- Run the baseline only sanity check: es binary 0.65 (chance 0.50); es multi 0.36 (chance 0.25); gwt binary 1.00 (chance 0.50); gwt multi 0.99 (chance 0.10)


## Colab: save the outputs
Colab deletes everything when the session ends. This zips `results/` and `report/` into `outputs_<notebook>.zip`,
copies it to your Google Drive (`MyDrive/gas-repr-guide/`) and downloads it. On a laptop it does nothing.

In [7]:
if IN_COLAB:
    out_zip = shutil.make_archive(f"/content/outputs_04f_guide_table", "zip", REPO, "results")
    shutil.make_archive(f"/content/report_04f_guide_table", "zip", REPO, "report")
    files_out = [out_zip, f"/content/report_04f_guide_table.zip"]
    print(files_out)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        out_dir = "/content/drive/MyDrive/gas-repr-guide"
        os.makedirs(out_dir, exist_ok=True)
        for f in files_out:
            shutil.copy(f, out_dir)
        print("copied to", out_dir)
    except Exception as e:
        print("Drive copy failed:", e)
    from google.colab import files
    for f in files_out:
        files.download(f)